In [1]:
!pip install optuna rank_bm25 sentence-transformers scikit-learn river matplotlib pyyaml tqdm


[notice] A new release of pip is available: 25.1.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
!mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json

The syntax of the command is incorrect.


In [3]:

# =============================================================================
#Supervised Auto-tuned kNN Retriever (Optuna)

# ── Cell 1: Imports & Config ──────────────────────────────────────────────────
import time, json, yaml, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

import optuna
from optuna.samplers import TPESampler
optuna.logging.set_verbosity(optuna.logging.WARNING)

from rank_bm25 import BM25Okapi
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import normalize
from sklearn.neighbors import NearestNeighbors
from sentence_transformers import SentenceTransformer

from river import linear_model, preprocessing, metrics, drift
from river.stream import iter_array

warnings.filterwarnings("ignore")
np.random.seed(42)

DEVICE      = "cpu"
EMBED_MODEL = "BAAI/bge-small-en-v1.5"
N_TRIALS    = 40          # Optuna trials (raise to 80+ for final run)
TOP_K_EVAL  = 5           # NDCG@5 / Recall@5
DATA_SUBSET = 500         # papers to use (max out at what the dataset has)

print("Imports done")

C:\Users\User\PycharmProjects\.venv1\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Imports done


In [ ]:
# ── Cell 2: Load Kaggle arXiv Dataset ────────────────────────────────────────
# Dataset: sumitm004/arxiv-scientific-research-papers-dataset
# Make sure you have your Kaggle API key set up in Colab:
#   1. Upload your kaggle.json to Colab files, then run:
#      !mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json

import kagglehub
import os

print("Downloading arXiv dataset from Kaggle …")
dataset_path = kagglehub.dataset_download(
    "sumitm004/arxiv-scientific-research-papers-dataset"
)
print(f"Downloaded to: {dataset_path}")

# ── Auto-find the CSV/parquet file inside the downloaded folder ───────────────
all_files = []
for root, dirs, files in os.walk(dataset_path):
    for f in files:
        all_files.append(os.path.join(root, f))

print(f"Files found: {all_files}")

# Pick the first CSV or parquet file found
data_file = next(
    (f for f in all_files if f.endswith((".csv", ".parquet", ".tsv"))), None
)
assert data_file, f"No CSV/parquet file found. Files present: {all_files}"
print(f"Loading: {data_file}")

raw_df = pd.read_csv(data_file) if data_file.endswith((".csv", ".tsv")) \
         else pd.read_parquet(data_file)
print(f"Raw dataset shape : {raw_df.shape}")
print(f"Columns           : {list(raw_df.columns)}")

# ── Normalise column names (lowercase + strip whitespace) ────────────────────
raw_df.columns = raw_df.columns.str.strip().str.lower().str.replace(" ", "_")

# ── Identify the abstract/text column ────────────────────────────────────────
# Common names in arXiv Kaggle datasets — adjust if yours differs
TEXT_COL     = next((c for c in raw_df.columns if c in
                     ["abstract", "summary", "description", "text"]), None)
TITLE_COL    = next((c for c in raw_df.columns if "title"    in c), None)
CAT_COL      = next((c for c in raw_df.columns if "categor"  in c), None)
AUTHOR_COL   = next((c for c in raw_df.columns if "author"   in c), None)
ID_COL       = next((c for c in raw_df.columns if c in
                     ["id", "arxiv_id", "paper_id"]), None)

print(f"\nDetected columns → text: '{TEXT_COL}' | title: '{TITLE_COL}' "
      f"| category: '{CAT_COL}' | id: '{ID_COL}'")

assert TEXT_COL, "Could not find an abstract/text column — check raw_df.columns above"

# ── Clean & filter ────────────────────────────────────────────────────────────
df = raw_df.dropna(subset=[TEXT_COL]).copy()
df[TEXT_COL] = df[TEXT_COL].str.strip()
df = df[df[TEXT_COL].str.len() > 80]           # drop very short abstracts

# Filter to AI/ML papers if category column exists
if CAT_COL:
    ai_mask = df[CAT_COL].str.contains(
        "cs.AI|cs.CL|cs.LG|cs.CV|stat.ML", na=False, case=False)
    df_ai = df[ai_mask]
    df = df_ai if len(df_ai) >= 100 else df    # fall back to full set if too small
    print(f"AI/ML papers found: {len(df_ai)} | Using: {len(df)}")

# Sample to DATA_SUBSET for speed
df = df.sample(min(DATA_SUBSET, len(df)), random_state=42).reset_index(drop=True)

# ── Build the lists the rest of the notebook expects ─────────────────────────
texts   = df[TEXT_COL].tolist()
doc_ids = (df[ID_COL].astype(str).tolist()
           if ID_COL else [f"doc_{i:04d}" for i in range(len(df))])
labels  = (df[CAT_COL].astype(str).tolist()
           if CAT_COL else ["unknown"] * len(df))

# ── Gold Q/A set: titles as queries, abstracts as corpus ─────────────────────
# Using the full abstract as both query AND document is trivially easy for
# dense embeddings (cosine sim = 1.0 always → NDCG = 1.0 always → nothing
# for Optuna to optimise).
#
# Instead: query = title (short, keyword-like)
#          corpus = abstract (longer, semantic)
# This mirrors real usage and is hard enough that hyperparams actually matter.

assert TITLE_COL, " No title column found — check your dataset columns"

GOLD_N = min(50, len(df) // 5)

# Build gold pairs: (title → doc_id) for first GOLD_N rows
gold_queries = df[TITLE_COL].iloc[:GOLD_N].fillna("").tolist()
gold_doc_ids = doc_ids[:GOLD_N]

# Corpus = ALL abstracts (gold docs included so retriever can find them)
train_texts  = texts        # abstracts
train_ids    = doc_ids
train_labels = labels

print(f"Gold queries (titles) : {GOLD_N}")
print(f"Corpus (abstracts)    : {len(train_texts)}")
print(f"\nExample query : '{gold_queries[0]}'")
print(f"Expected doc  : '{gold_doc_ids[0]}'")

# ── Quick sanity-check preview ────────────────────────────────────────────────
print(f"\nCorpus ready: {len(train_texts)} docs | Gold queries: {GOLD_N}")
print(f"\nSample abstract:\n  {train_texts[0][:200]} …")
if TITLE_COL:
    print(f"Sample title    :\n  {df[TITLE_COL].iloc[GOLD_N]}")

📥 Downloading arXiv dataset from Kaggle …


100%|██████████| 62.4M/62.4M [00:02<00:00, 27.9MB/s]

Extracting files...


Downloaded to: /root/.cache/kagglehub/datasets/sumitm004/arxiv-scientific-research-papers-dataset/versions/2
Files found: ['/root/.cache/kagglehub/datasets/sumitm004/arxiv-scientific-research-papers-dataset/versions/2/arXiv_scientific dataset.csv']
✅ Loading: /root/.cache/kagglehub/datasets/sumitm004/arxiv-scientific-research-papers-dataset/versions/2/arXiv_scientific dataset.csv
Raw dataset shape : (136238, 10)
Columns           : ['id', 'title', 'category', 'category_code', 'published_date', 'updated_date', 'authors', 'first_author', 'summary', 'summary_word_count']

🔍 Detected columns → text: 'summary' | title: 'title' | category: 'category' | id: 'id'
AI/ML papers found: 0 | Using: 136206
Gold queries (titles) : 50
Corpus (abstracts)    : 500

Example query : 'Multi-hop assortativities for networks classification'
Expected doc  : 'abs-1809.06253v2'

✅ Corpus ready: 500 docs | Gold queries: 50

Sample abstract:
  Several social, medical, engineering and biological challenges rely on

In [ ]:
#  ── Cell 3: Embeddings ────────────────────────────────────────────────────────
print("Encoding dense embeddings …")
encoder = SentenceTransformer(EMBED_MODEL, device=DEVICE)

corpus_emb = encoder.encode(train_texts, batch_size=64,
                             show_progress_bar=True, normalize_embeddings=True)
# Gold queries are now titles (short), corpus is abstracts (long) — intentionally asymmetric
query_emb  = encoder.encode(gold_queries, batch_size=64,
                             show_progress_bar=True, normalize_embeddings=True)

print(f"Embedding shape: {corpus_emb.shape}")


Encoding dense embeddings …


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/8 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embedding shape: (500, 384)


In [ ]:
#  ── Cell 4: BM25 Index ───────────────────────────────────────────────────────
tokenized = [t.lower().split() for t in train_texts]
bm25      = BM25Okapi(tokenized)

def bm25_scores(query: str) -> np.ndarray:
    """Return normalised BM25 scores for all corpus docs."""
    raw_s = np.array(bm25.get_scores(query.lower().split()))
    mx = raw_s.max()
    return raw_s / mx if mx > 0 else raw_s

print("✅ BM25 index built")

✅ BM25 index built
